In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import joblib

# 1. Load Data
# Assuming 'processed_kaggle_sepsis.parquet' from Notebook 1
df = pd.read_parquet('processed_kaggle_sepsis.parquet')

In [2]:
# 2. Define Features based on your Kaggle snippet
dynamic_cols = [
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2',
    'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN',
    'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct',
    'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium',
    'Bilirubin_total', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets'
]
static_cols = ['Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime']

In [3]:
# 3. Paper Step: Forward Filling (Clinical Stability)
# We group by Patient_ID to ensure data doesn't leak between different patients
df[dynamic_cols] = df.groupby('Patient_ID')[dynamic_cols].ffill().fillna(0)

In [4]:
# 4. Paper Step: Normalization
scaler = StandardScaler()
df[dynamic_cols + ['Age', 'HospAdmTime']] = scaler.fit_transform(df[dynamic_cols + ['Age', 'HospAdmTime']])

In [5]:
# 5. Paper Step: PyTorch Dataset Construction
# This replicates the "Sliding-window sequence prediction setup" from the paper
class SepsisDataset(Dataset):
    def __init__(self, dataframe, dynamic_features, static_features, max_len=72):
        self.p_ids = dataframe['Patient_ID'].unique()
        self.df = dataframe
        self.dyn_cols = dynamic_features
        self.stat_cols = static_features
        self.max_len = max_len

    def __len__(self):
        return len(self.p_ids)

    def __getitem__(self, idx):
        patient_data = self.df[self.df['Patient_ID'] == self.p_ids[idx]]
        
        # Dynamic Sequence (Time, Features)
        dyn_values = patient_data[self.dyn_cols].values
        
        # Static Context (Features,)
        stat_values = patient_data[self.stat_cols].values[0]
        
        # Labels (Time,)
        labels = patient_data['SepsisLabel'].values
        
        # Causal Padding (Pre-padding to max_len)
        curr_len = len(dyn_values)
        if curr_len > self.max_len:
            dyn_values = dyn_values[:self.max_len]
            labels = labels[:self.max_len]
            mask = np.ones(self.max_len)
        else:
            pad_size = self.max_len - curr_len
            dyn_values = np.pad(dyn_values, ((pad_size, 0), (0, 0)), mode='constant')
            labels = np.pad(labels, (pad_size, 0), mode='constant')
            # Mask to ignore padded values during loss calculation
            mask = np.concatenate([np.zeros(pad_size), np.ones(curr_len)])

        return {
            'dynamic': torch.tensor(dyn_values, dtype=torch.float32),
            'static': torch.tensor(stat_values, dtype=torch.float32),
            'label': torch.tensor(labels, dtype=torch.float32),
            'mask': torch.tensor(mask, dtype=torch.float32)
        }

# 6. Initialize DataLoader
max_seq_length = 72 # 72 hours window
dataset = SepsisDataset(df, dynamic_cols, static_cols, max_len=max_seq_length)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 7. Save Preprocessing Artifacts
preprocessing_artifacts = {
    'scaler': scaler,
    'dynamic_cols': dynamic_cols,
    'static_cols': static_cols,
    'max_seq_length': max_seq_length
}
joblib.dump(preprocessing_artifacts, 'preprocessing_artifacts_CNN_LSTM.pkl')

print(f"Preprocessing complete.")
print(f"Dataset size: {len(dataset)} patients")
# Peek at one batch
batch = next(iter(train_loader))
print(f"Dynamic Batch Shape: {batch['dynamic'].shape}") # (Batch, Time, Features)
print(f"Static Batch Shape: {batch['static'].shape}")   # (Batch, Features)

Preprocessing complete.
Dataset size: 40336 patients
Dynamic Batch Shape: torch.Size([32, 72, 33])
Static Batch Shape: torch.Size([32, 5])
